# Walkthrough: weekday-seasonal forecaster

This notebook shows the pieces behind `scripts/run_pipeline.py`: the synthetic data, what the seasonal model learns, and the backtest results. All core logic lives in `src/`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
from weekday_seasonal_forecaster import generate_daily_flows, SeasonalWeekday, NaiveWeekly, MovingAverage
from weekday_seasonal_forecaster.data import TRUE_WEEKDAY_FACTORS
from weekday_seasonal_forecaster.evaluate import walk_forward, results_table

## 1. The data
Volume is the product of known components, which are kept as columns.

In [ ]:
df = generate_daily_flows('2021-01-01', '2025-12-31', seed=42)
df.head(10)

In [ ]:
df.set_index('date')['volume'].loc['2025-09':'2025-12'].plot(figsize=(11, 3), title='Daily volume, Sep to Dec 2025');

## 2. What the seasonal model learns
The weekday factors estimated from the last 12 weeks, next to the true factors used to simulate the data.

In [ ]:
model = SeasonalWeekday().fit(df)
pd.DataFrame({
    'estimated': model.weekday_factors_.values,
    'true': TRUE_WEEKDAY_FACTORS,
}, index=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']).round(3)

In [ ]:
model.month_factors_.round(3)  # payday buckets: 1-3, 4-13, 14-16, 17 to month end, last 3 days

In [ ]:
model.predict(14)  # next two weeks

## 3. Walk-forward backtest
At each origin the model is refit on data up to that day only, then forecasts 1, 7 and 30 days ahead.

In [ ]:
bt = pd.concat([walk_forward(df, m) for m in (NaiveWeekly, MovingAverage, SeasonalWeekday)])
res = results_table(bt)
res.assign(**{c: (res[c] / 1e6).round(2) for c in ['MAE', 'RMSE', 'required_buffer']}).drop(columns='MSE')